# Library Imports and Device Setup

Here, we import all the necessary libraries and modules required for our chatbot project using the Cornell Movie Dialog Corpus. The imported modules include:

- **PyTorch & TorchScript**: For building and optimizing our neural network models.
- **PyTorch's Neural Network (nn) & Optimizers (optim)**: To define layers, activation functions, and optimization algorithms.
- **Torch Functional (F)**: For stateless functions like activation and loss functions.
- **CSV, JSON, and IO modules**: For file handling and data processing.
- **Regular Expressions (re), Unicode utilities, and Codecs**: To pre-process text data.
- **Itertools, Math, and NumPy**: For various utility functions and numerical operations.

We also configure the computation device. If a hardware accelerator (like a GPU) is available, it is used; otherwise, the code falls back to the CPU. This setup ensures that our code runs efficiently regardless of the available hardware.

Let's now dive into the code that 


In [2]:
import torch
from torch.jit import script, trace
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
import csv
import random
import re
import os
import unicodedata
import codecs
from io import open
import itertools
import math
import json
import numpy as np
#If a hardware accelerator (like a GPU) is available, it will be used to speed up computations; otherwise, the CPU will be used.
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


# Reading and Previewing the Movie Lines File

Here, we demonstrate how to open and read from the `movie_lines.txt` file, which is part of our movie corpus dataset. The code constructs the file path in a platform-independent way, opens the file with the proper encoding, and prints the first 10 lines to the console. This helps in verifying the file content and ensuring that the file is read correctly.


In [3]:
# Constructing the file path to movie_lines.txt inside the movie-corpus folder.
file_path = os.path.join("movie-corpus", "movie_lines.txt")

# Opening and printing the first 10 lines to inspect the file content.
with open(file_path, 'r', encoding='iso-8859-1') as f:
    for i, line in enumerate(f):
        if i < 10:
            print(line.strip())
        else:
            break


L1045 +++$+++ u0 +++$+++ m0 +++$+++ BIANCA +++$+++ They do not!
L1044 +++$+++ u2 +++$+++ m0 +++$+++ CAMERON +++$+++ They do to!
L985 +++$+++ u0 +++$+++ m0 +++$+++ BIANCA +++$+++ I hope so.
L984 +++$+++ u2 +++$+++ m0 +++$+++ CAMERON +++$+++ She okay?
L925 +++$+++ u0 +++$+++ m0 +++$+++ BIANCA +++$+++ Let's go.
L924 +++$+++ u2 +++$+++ m0 +++$+++ CAMERON +++$+++ Wow
L872 +++$+++ u0 +++$+++ m0 +++$+++ BIANCA +++$+++ Okay -- you're gonna need to learn how to lie.
L871 +++$+++ u2 +++$+++ m0 +++$+++ CAMERON +++$+++ No
L870 +++$+++ u0 +++$+++ m0 +++$+++ BIANCA +++$+++ I'm kidding.  You know how sometimes you just become this "persona"?  And you don't know how to quit?
L869 +++$+++ u0 +++$+++ m0 +++$+++ BIANCA +++$+++ Like my fear of wearing pastels?


# Loading and Processing the Movie Dialog Corpus

Here, we define several functions to load and process the movie dialog corpus:

- **`loadLines(file_path)`**:  
  Reads `movie_lines.txt` and returns a dictionary mapping each line ID to its text.

- **`loadConversations(file_path)`**:  
  Reads `movie_conversations.txt` and returns a list of conversations. Each conversation is represented as a list of line IDs.

- **`extractSentencePairs(lines_dict, conversations)`**:  
  Uses the loaded lines and conversations to create pairs of consecutive lines (input and response) suitable for training a chatbot.

We then construct the file paths, load the data, and extract the conversation pairs. Finally, the total number of extracted conversation pairs is printed for verification.


In [4]:

def loadLines(file_path):
    """
    Reads movie_lines.txt and returns a dictionary that maps
    each line ID to its text.
    """
    lines = {}
    with open(file_path, 'r', encoding='iso-8859-1') as f:
        for line in f:
            # Each line is separated by the delimiter " +++$+++ "
            parts = line.strip().split(" +++$+++ ")
            # Expected format: [lineID, speaker, movieID, character name, text]
            if len(parts) == 5:
                line_id = parts[0]
                text = parts[4]
                lines[line_id] = text
    return lines

def loadConversations(file_path):
    """
    Reads movie_conversations.txt and returns a list of conversations.
    Each conversation is a list of line IDs.
    
    The expected format of each line in movie_conversations.txt is:
    [character1ID, character2ID, movieID, "[lineID1, lineID2, ...]"]
    """
    conversations = []
    with open(file_path, 'r', encoding='iso-8859-1') as f:
        for line in f:
            parts = line.strip().split(" +++$+++ ")
            # We expect 4 parts per line
            if len(parts) == 4:
                # The last part is a string representation of a list of line IDs.
                line_ids_str = parts[3]
                # Convert the string to an actual list using eval (or use a safer alternative)
                line_ids = eval(line_ids_str)
                conversations.append(line_ids)
    return conversations

def extractSentencePairs(lines_dict, conversations):
    """
    Given a dictionary of lineID to text and a list of conversations (which are lists of lineIDs),
    this function creates pairs of consecutive lines as (input, response) pairs.
    """
    qa_pairs = []
    for conv in conversations:
        # Loop through each conversation, pairing each line with its following line.
        for i in range(len(conv) - 1):
            input_line = lines_dict.get(conv[i], "")
            target_line = lines_dict.get(conv[i+1], "")
            # Only add pairs if both lines are non-empty.
            if input_line and target_line:
                qa_pairs.append([input_line, target_line])
    return qa_pairs

# Construct file paths for the dataset files in your "movie-corpus" folder.
lines_path = os.path.join("movie-corpus", "movie_lines.txt")
conv_path = os.path.join("movie-corpus", "movie_conversations.txt")

# Load the lines and conversations.
lines_dict = loadLines(lines_path)
conversations = loadConversations(conv_path)

# Extract input-response pairs.
sentence_pairs = extractSentencePairs(lines_dict, conversations)

print("Total conversation pairs extracted:", len(sentence_pairs))


Total conversation pairs extracted: 221282


In [4]:

def clean_text(text):
    """
    Lowercases, removes non-alphanumeric characters, and extra spaces from a given text.
    """
    text = text.lower()
    # Remove any character that is not a letter, number, or whitespace.
    text = re.sub(r"[^a-z0-9\s]", "", text)
    # Replace multiple spaces with a single space.
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Example: Clean all sentence pairs.
clean_pairs = []
for pair in sentence_pairs:
    input_line = clean_text(pair[0])
    target_line = clean_text(pair[1])
    clean_pairs.append([input_line, target_line])


In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Combine all sentences (both inputs and responses) to build a comprehensive vocabulary.
all_sentences = [sentence for pair in clean_pairs for sentence in pair]

# Initialize and fit the tokenizer.
tokenizer = Tokenizer()
tokenizer.fit_on_texts(all_sentences)

# Determine the vocabulary size.
vocab_size = len(tokenizer.word_index) + 1  # +1 for any padding index
print("Vocabulary Size:", vocab_size)


Vocabulary Size: 65576


In [8]:
# Split your pairs into inputs and targets.
input_texts = [pair[0] for pair in clean_pairs]
target_texts = [pair[1] for pair in clean_pairs]

# Convert texts to sequences of integers.
input_sequences = tokenizer.texts_to_sequences(input_texts)
target_sequences = tokenizer.texts_to_sequences(target_texts)


In [9]:
# Determine maximum sequence lengths.
max_len_input = max(len(seq) for seq in input_sequences)
max_len_target = max(len(seq) for seq in target_sequences)

# Pad sequences so that all sequences in a batch have the same length.
input_sequences = pad_sequences(input_sequences, maxlen=max_len_input, padding='post')
target_sequences = pad_sequences(target_sequences, maxlen=max_len_target, padding='post')

print("Input sequence shape:", input_sequences.shape)
print("Target sequence shape:", target_sequences.shape)


Input sequence shape: (221282, 313)
Target sequence shape: (221282, 552)


In [11]:
# Example parameters
embedding_dim = 256   # Size of the embedding vectors
latent_dim = 512      # Size of the LSTM's hidden state
vocab_size = vocab_size  # From your tokenizer (make sure this is defined from your preprocessing)


In [12]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense

# Define the encoder input layer
encoder_inputs = Input(shape=(max_len_input,), name='encoder_input')

# Add an Embedding layer to transform word indices into dense vectors.
encoder_embedding = Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True, name='encoder_embedding')(encoder_inputs)

# Add an LSTM layer. We only need the final hidden and cell states for the decoder.
encoder_lstm = LSTM(latent_dim, return_state=True, name='encoder_lstm')
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)

# Store the encoder's final states (to be used as initial states for the decoder).
encoder_states = [state_h, state_c]


In [13]:
# Define the decoder input layer.
decoder_inputs = Input(shape=(max_len_target,), name='decoder_input')

# The decoder also has an Embedding layer.
decoder_embedding_layer = Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True, name='decoder_embedding')
decoder_embedding = decoder_embedding_layer(decoder_inputs)

# The decoder LSTM receives its initial state from the encoder.
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, name='decoder_lstm')
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

# A Dense layer with softmax activation produces a probability distribution over the vocabulary for each time step.
decoder_dense = Dense(vocab_size, activation='softmax', name='decoder_dense')
decoder_outputs = decoder_dense(decoder_outputs)


In [14]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

# Compile the model.
# We're using 'sparse_categorical_crossentropy' as our loss function because our targets are integer encoded.
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

# Print the model summary to inspect the architecture.
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ encoder_input (InputLayer)    │ (None, 313)               │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_input (InputLayer)    │ (None, 552)               │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ encoder_embedding (Embedding) │ (None, 313, 256)          │      16,787,456 │ encoder_input[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ not_equal (NotEqual)          │ (None, 313)               │               0 │ encoder_input[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_embedding (Embedding) │ (None, 552, 256)          │      16,787,456 │ decoder_input[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ encoder_lstm (LSTM)           │ [(None, 512), (None,      │       1,574,912 │ encoder_embedding[0][0],   │
│                               │ 512), (None, 512)]        │                 │ not_equal[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_lstm (LSTM)           │ [(None, 552, 512), (None, │       1,574,912 │ decoder_embedding[0][0],   │
│                               │ 512), (None, 512)]        │                 │ encoder_lstm[0][1],        │
│                               │                           │                 │ encoder_lstm[0][2]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_dense (Dense)         │ (None, 552, 65576)        │      33,640,488 │ decoder_lstm[0][0]         │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 70,365,224 (268.42 MB)

 Trainable params: 70,365,224 (268.42 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
# Example: Adding special tokens to each target sentence.
start_token = '<start>'
end_token = '<end>'

# Modify each target sentence to include start and end tokens.
target_texts_with_tokens = [start_token + ' ' + text + ' ' + end_token for text in target_texts]


In [16]:
# Split the target sentences into decoder input and target sequences.
decoder_input_texts = [ ' '.join(text.split()[:-1]) for text in target_texts_with_tokens ]
decoder_target_texts = [ ' '.join(text.split()[1:]) for text in target_texts_with_tokens ]


In [17]:
# Convert decoder input and target texts to sequences.
decoder_input_sequences = tokenizer.texts_to_sequences(decoder_input_texts)
decoder_target_sequences = tokenizer.texts_to_sequences(decoder_target_texts)


In [18]:


# Pad the decoder input and target sequences.
decoder_input_sequences = pad_sequences(decoder_input_sequences, maxlen=max_len_target, padding='post')
decoder_target_sequences = pad_sequences(decoder_target_sequences, maxlen=max_len_target, padding='post')

# For sparse_categorical_crossentropy, the target data needs to have an extra dimension.
decoder_target_sequences = np.expand_dims(decoder_target_sequences, -1)


In [21]:
import numpy as np
import pickle

# Suppose you already have these arrays from preprocessing:
# input_sequences, decoder_input_sequences, decoder_target_sequences

# Define how many subsets you want.
num_subsets = 300

# Split the arrays into subsets. This creates lists where each element is a subset.
input_subsets = np.array_split(input_sequences, num_subsets)
decoder_input_subsets = np.array_split(decoder_input_sequences, num_subsets)
decoder_target_subsets = np.array_split(decoder_target_sequences, num_subsets)

# Save each subset to a separate file.
for i in range(num_subsets):
    subset_data = {
        "encoder": input_subsets[i],
        "decoder_input": decoder_input_subsets[i],
        "decoder_target": decoder_target_subsets[i]
    }
    # Save to a file named subset_1.pkl, subset_2.pkl, etc.
    with open(f"subset_{i+1}.pkl", "wb") as f:
        pickle.dump(subset_data, f)
    print(f"Saved subset {i+1} with {input_subsets[i].shape[0]} samples.")


Saved subset 1 with 738 samples.
Saved subset 2 with 738 samples.
Saved subset 3 with 738 samples.
Saved subset 4 with 738 samples.
Saved subset 5 with 738 samples.
Saved subset 6 with 738 samples.
Saved subset 7 with 738 samples.
Saved subset 8 with 738 samples.
Saved subset 9 with 738 samples.
Saved subset 10 with 738 samples.
Saved subset 11 with 738 samples.
Saved subset 12 with 738 samples.
Saved subset 13 with 738 samples.
Saved subset 14 with 738 samples.
Saved subset 15 with 738 samples.
Saved subset 16 with 738 samples.
Saved subset 17 with 738 samples.
Saved subset 18 with 738 samples.
Saved subset 19 with 738 samples.
Saved subset 20 with 738 samples.
Saved subset 21 with 738 samples.
Saved subset 22 with 738 samples.
Saved subset 23 with 738 samples.
Saved subset 24 with 738 samples.
Saved subset 25 with 738 samples.
Saved subset 26 with 738 samples.
Saved subset 27 with 738 samples.
Saved subset 28 with 738 samples.
Saved subset 29 with 738 samples.
Saved subset 30 with 73

In [ ]:
import pickle

# Define which subset you want to train on (e.g., subset 1)
subset_index = 1  # For subset_1.pkl (set this to 1, 2, ..., num_subsets)

# Load the subset from file.
with open(f"subset_{subset_index}.pkl", "rb") as f:
    subset_data = pickle.load(f)

# Extract the data from the loaded dictionary.
encoder_subset = subset_data["encoder"]
decoder_input_subset = subset_data["decoder_input"]
decoder_target_subset = subset_data["decoder_target"]

# Now train your model on this subset.
batch_size = 16
epochs_for_subset = 3  # Train for a desired number of epochs on this subset

model.fit(
    [encoder_subset, decoder_input_subset],
    decoder_target_subset,
    batch_size=batch_size,
    epochs=epochs_for_subset,
    validation_split=0.2
)

# Optionally, save a checkpoint of the model after training on this subset.
model.save(f"model_after_subset_{subset_index}.h5")


Epoch 1/3
 2/37 ━━━━━━━━━━━━━━━━━━━━ 30:32:18 3141s/step - loss: 11.0908